# Bronze processing

Read preserved raw Bundle envelopes, explode resource entries, and append resource-level Delta records with source and ingestion lineage. Watermark commit is deliberately gated on an explicit batch_started_at value.

In [ ]:
resource_type = "Patient"
batch_id = ""  # optional: process one batch; empty = all available raw batches

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime, timezone

allowed = {"Patient", "Encounter", "Observation", "Condition"}
if resource_type not in allowed:
    raise ValueError(f"Unsupported resource_type: {resource_type}")
raw_glob = f"Files/raw/fhir/{resource_type.lower()}/extraction_date=*/batch_id=*/*.json"
raw_df = spark.read.option("multiLine", "true").json(raw_glob)
if batch_id:
    raw_df = raw_df.filter(F.col("ingestion_metadata.batch_id") == batch_id)

entries = raw_df.select(
    F.col("ingestion_metadata.resource_type").alias("resource_type"),
    F.col("ingestion_metadata.batch_id").alias("batch_id"),
    F.col("ingestion_metadata.extraction_timestamp").alias("extraction_timestamp"),
    F.col("ingestion_metadata.api_url_or_params").alias("api_url_or_params"),
    F.explode_outer(F.col("fhir_bundle.entry")).alias("entry")
).filter(F.col("entry.resource").isNotNull())

bronze_df = entries.select(
    "resource_type",
    F.col("entry.resource.id").cast("string").alias("resource_id"),
    F.col("entry.resource.meta.versionId").cast("string").alias("source_version_id"),
    F.col("entry.resource.meta.lastUpdated").cast("string").alias("source_last_updated"),
    "extraction_timestamp",
    F.current_timestamp().cast("string").alias("ingestion_timestamp"),
    "api_url_or_params",
    "batch_id",
    F.to_json(F.col("entry.resource")).alias("raw_resource_json")
).filter(F.col("resource_id").isNotNull())

table_name = f"bronze_{resource_type.lower()}"
if bronze_df.limit(1).count() > 0:
    (bronze_df.write.format("delta").mode("append")
        .option("mergeSchema", "true").saveAsTable(table_name))
    print(f"Appended Bronze records to {table_name}")
else:
    print(f"No resource rows found for {resource_type}; no watermark update performed.")

# Commit a safe run watermark only after raw + Bronze processing succeeds.
# The pipeline should pass batch_started_at as the watermark candidate captured before API calls.
batch_started_at = ""
if batch_started_at and bronze_df.limit(1).count() > 0:
    candidate = spark.createDataFrame(
        [(resource_type, batch_started_at, datetime.now(timezone.utc).isoformat(), batch_id)],
        "resource_type string, watermark_utc string, updated_at string, batch_id string"
    )
    if spark.catalog.tableExists("metadata_watermark"):
        target = DeltaTable.forName(spark, "metadata_watermark")
        (target.alias("t").merge(candidate.alias("s"), "t.resource_type = s.resource_type")
         .whenMatchedUpdate(set={"watermark_utc":"s.watermark_utc","updated_at":"s.updated_at","batch_id":"s.batch_id"})
         .whenNotMatchedInsertAll().execute())
    else:
        candidate.write.format("delta").mode("overwrite").saveAsTable("metadata_watermark")
    print("Watermark committed after Bronze processing.")
else:
    print("Watermark not changed. Supply batch_started_at from the ingestion run after validating the batch.")
